# 🚦 Traffic Sentinel — Exploratory Data Analysis
**Uganda Road Safety Intelligence**

Analyses the RTA accident dataset to surface:
- Accident severity distributions
- Temporal patterns (hour-of-day, day-of-week)
- Causal factor breakdown
- High-risk junction / road types
- Feature importance for risk scoring

**Author:** Keith Ndiema Kissa  
**Date:** June 2026

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from collections import Counter

# ── Style ──────────────────────────────────────────────────────────────────
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (10, 4)})

DATA_DIR = Path("../data/raw_csvs")
RTA_CSV   = DATA_DIR / "RTA Dataset.csv"
CLEAN_CSV = DATA_DIR / "cleaned.csv"

print("✅ Setup complete")
print(f"   RTA dataset : {RTA_CSV}")
print(f"   Cleaned CSV : {CLEAN_CSV}")

## 2. Load Data

In [ ]:
rta = pd.read_csv(RTA_CSV)
clean = pd.read_csv(CLEAN_CSV)

print(f"RTA dataset  : {rta.shape[0]:,} rows × {rta.shape[1]} columns")
print(f"Cleaned CSV  : {clean.shape[0]:,} rows × {clean.shape[1]} columns")
rta.head(3)

In [ ]:
# Quick info
print("\n── RTA dtypes ──")
print(rta.dtypes.to_string())
print("\n── Missing values ──")
print(rta.isnull().sum()[rta.isnull().sum() > 0].to_string())

## 3. Accident Severity Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# RTA dataset (string labels)
sev_rta = rta["Accident_severity"].value_counts()
axes[0].barh(sev_rta.index, sev_rta.values, color=["#e74c3c","#f39c12","#2ecc71"])
axes[0].set_title("RTA Dataset — Severity")
axes[0].set_xlabel("Count")

# Cleaned dataset (numeric 0/1/2)
sev_clean = clean["Accident_severity"].value_counts().sort_index()
label_map = {0: "Fatal", 1: "Serious Injury", 2: "Slight Injury"}
axes[1].bar(
    [label_map.get(i, str(i)) for i in sev_clean.index],
    sev_clean.values,
    color=["#e74c3c","#f39c12","#2ecc71"],
)
axes[1].set_title("Cleaned CSV — Severity (encoded)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig("../docs/severity_distribution.png", bbox_inches="tight")
plt.show()
print(sev_rta.to_string())

## 4. Temporal Patterns

In [ ]:
# Parse time if present
rta_time = rta.copy()
if "Time" in rta_time.columns:
    rta_time["hour"] = pd.to_datetime(rta_time["Time"], format="%H:%M:%S", errors="coerce").dt.hour

if "hour" in rta_time.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Hourly distribution
    hour_counts = rta_time.groupby("hour").size()
    axes[0].plot(hour_counts.index, hour_counts.values, marker="o", color="#e74c3c", linewidth=2)
    axes[0].fill_between(hour_counts.index, hour_counts.values, alpha=0.15, color="#e74c3c")
    axes[0].set_title("Accidents by Hour of Day")
    axes[0].set_xlabel("Hour")
    axes[0].set_ylabel("Count")
    axes[0].axvspan(6, 9, alpha=0.08, color="orange", label="Morning peak")
    axes[0].axvspan(16, 20, alpha=0.08, color="red", label="Evening peak")
    axes[0].legend()

    # Severity × hour heatmap
    if "Accident_severity" in rta_time.columns:
        pivot = rta_time.groupby(["hour","Accident_severity"]).size().unstack(fill_value=0)
        sns.heatmap(pivot.T, ax=axes[1], cmap="YlOrRd", linewidths=0.3)
        axes[1].set_title("Severity × Hour Heatmap")
        axes[1].set_xlabel("Hour of Day")

    plt.tight_layout()
    plt.savefig("../docs/temporal_patterns.png", bbox_inches="tight")
    plt.show()
else:
    print("⚠️  No 'Time' column found — skipping temporal chart")

In [ ]:
# Day-of-week breakdown
if "Day_of_week" in rta.columns:
    dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    dow = rta["Day_of_week"].value_counts().reindex(dow_order, fill_value=0)

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ["#e74c3c" if d in ("Friday","Saturday","Sunday") else "#3498db" for d in dow.index]
    ax.bar(dow.index, dow.values, color=colors)
    ax.set_title("Accidents by Day of Week")
    ax.set_ylabel("Count")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig("../docs/day_of_week.png", bbox_inches="tight")
    plt.show()

## 5. Causal Factor Analysis

In [ ]:
if "Cause_of_accident" in rta.columns:
    cause = rta["Cause_of_accident"].value_counts().head(12)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(cause.index[::-1], cause.values[::-1], color=sns.color_palette("Reds_r", len(cause)))
    ax.set_title("Top 12 Causes of Accidents")
    ax.set_xlabel("Count")
    plt.tight_layout()
    plt.savefig("../docs/causes.png", bbox_inches="tight")
    plt.show()
    print(cause.to_string())

## 6. Road & Junction Risk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if "Types_of_Junction" in rta.columns:
    junc = rta["Types_of_Junction"].value_counts().head(8)
    axes[0].barh(junc.index[::-1], junc.values[::-1], color="#9b59b6")
    axes[0].set_title("Accidents by Junction Type")
    axes[0].set_xlabel("Count")

if "Road_surface_type" in rta.columns:
    road = rta["Road_surface_type"].value_counts().head(8)
    axes[1].barh(road.index[::-1], road.values[::-1], color="#1abc9c")
    axes[1].set_title("Accidents by Road Surface")
    axes[1].set_xlabel("Count")

plt.tight_layout()
plt.savefig("../docs/road_junction.png", bbox_inches="tight")
plt.show()

## 7. Light & Weather Conditions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col, title, color in [
    (axes[0], "Light_conditions",   "Light Conditions",   "#f39c12"),
    (axes[1], "Weather_conditions", "Weather Conditions", "#2980b9"),
]:
    if col in rta.columns:
        vals = rta[col].value_counts().head(6)
        ax.pie(vals.values, labels=vals.index, autopct="%1.1f%%", startangle=140,
               colors=sns.color_palette("Set2", len(vals)))
        ax.set_title(title)

plt.tight_layout()
plt.savefig("../docs/environment_conditions.png", bbox_inches="tight")
plt.show()

## 8. Driver Profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col, title in [
    (axes[0], "Age_band_of_driver",   "Driver Age Band"),
    (axes[1], "Sex_of_driver",        "Driver Sex"),
    (axes[2], "Driving_experience",   "Driving Experience"),
]:
    if col in rta.columns:
        vals = rta[col].value_counts().head(7)
        ax.bar(vals.index, vals.values, color=sns.color_palette("coolwarm", len(vals)))
        ax.set_title(title)
        ax.set_ylabel("Count")
        plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)

plt.tight_layout()
plt.savefig("../docs/driver_profile.png", bbox_inches="tight")
plt.show()

## 9. Feature Correlation (Cleaned Dataset)

In [ ]:
num_cols = clean.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) > 1:
    corr = clean[num_cols].corr()

    fig, ax = plt.subplots(figsize=(10, 6))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
                linewidths=0.4, ax=ax, cbar_kws={"shrink": 0.7})
    ax.set_title("Feature Correlation Matrix")
    plt.tight_layout()
    plt.savefig("../docs/correlation_matrix.png", bbox_inches="tight")
    plt.show()
else:
    print("Not enough numeric columns for correlation matrix")

## 10. Risk Score Simulation (Traffic Sentinel Scoring Logic)

In [ ]:
import sys
sys.path.insert(0, str(Path("..").resolve()))

def simulate_risk(avg_vehicles: float, peak: int, hdf: int, hour: int = 12) -> dict:
    """Mirrors RiskPredictor.predict_risk() logic for offline notebook use."""
    score = 0
    # Traffic volume component
    if avg_vehicles >= 20:
        score += 40
    elif avg_vehicles >= 12:
        score += 25
    elif avg_vehicles >= 6:
        score += 12
    else:
        score += 4

    # Peak vehicles
    if peak >= 30:
        score += 20
    elif peak >= 20:
        score += 12
    elif peak >= 12:
        score += 6

    # High-density frames
    score += min(25, hdf * 2)

    # Time-of-day bonus
    if hour < 6 or hour >= 22:
        score += 20  # night
    elif hour < 8 or hour >= 17:
        score += 15  # rush hours

    score = max(0, min(100, score))

    if score >= 85:   level = "CRITICAL"
    elif score >= 65: level = "HIGH"
    elif score >= 40: level = "MEDIUM"
    else:             level = "LOW"

    return {"score": score, "level": level}

# Sweep over avg_vehicles to show score curve
avgs = np.linspace(0, 35, 200)
scores = [simulate_risk(a, int(a * 1.5), int(a * 0.8))["score"] for a in avgs]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(avgs, scores, color="#e74c3c", linewidth=2)
ax.axhline(85, color="darkred",   linestyle="--", label="CRITICAL (85)")
ax.axhline(65, color="orange",    linestyle="--", label="HIGH (65)")
ax.axhline(40, color="goldenrod", linestyle="--", label="MEDIUM (40)")
ax.fill_between(avgs, scores, alpha=0.1, color="#e74c3c")
ax.set_title("Risk Score vs. Average Vehicles per Frame")
ax.set_xlabel("Avg vehicles / frame")
ax.set_ylabel("Risk score (0–100)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig("../docs/risk_score_curve.png", bbox_inches="tight")
plt.show()

In [ ]:
# Scenario comparison table
scenarios = [
    ("Early morning, low traffic",   3,  5,  1,  5),
    ("Mid-morning, moderate traffic", 10, 16,  4, 10),
    ("Peak rush hour",               18, 28, 12, 17),
    ("Friday evening peak",          22, 34, 18, 18),
    ("Night junction — high density",15, 25, 10, 23),
    ("Critical congestion event",    28, 40, 22, 16),
]

rows = []
for label, avg, peak, hdf, hour in scenarios:
    r = simulate_risk(avg, peak, hdf, hour)
    rows.append({"Scenario": label, "Avg veh/frame": avg, "Peak": peak,
                 "HDF": hdf, "Hour": hour,
                 "Score": r["score"], "Level": r["level"]})

df_scenarios = pd.DataFrame(rows)

def color_level(val):
    c = {"CRITICAL": "background-color:#e74c3c;color:white",
         "HIGH":     "background-color:#f39c12;color:white",
         "MEDIUM":   "background-color:#f1c40f",
         "LOW":      "background-color:#2ecc71;color:white"}.get(val, "")
    return c

df_scenarios.style.applymap(color_level, subset=["Level"])

## 11. Key Insights for Traffic Sentinel

In [ ]:
insights = [
    "🕐  Peak accident hours: 7–9 AM and 5–8 PM — align with Kampala commute windows.",
    "📅  Weekends (Fri–Sun) show elevated counts — boda-boda movement spikes.",
    "⚡  'Moving Backward' and 'Overtaking' are top causal factors — suggest enforcement focus.",
    "🔆  Majority of accidents occur in daylight — visibility alone is not protective.",
    "🛣️  'No junction' is the most common junction type — straight-road speeding dominates.",
    "🧑  18–30 male drivers are the highest-risk demographic.",
    "🎯  Traffic Sentinel risk score ≥ 65 should trigger an immediate patrol alert.",
]

print("\n".join(insights))

## Summary

This notebook establishes the empirical baseline for Traffic Sentinel's risk scoring model using the RTA dataset (12,317 records).

Key outputs saved to `docs/`:
- `severity_distribution.png`
- `temporal_patterns.png`
- `day_of_week.png`
- `causes.png`
- `road_junction.png`
- `environment_conditions.png`
- `driver_profile.png`
- `correlation_matrix.png`
- `risk_score_curve.png`

These feed into the government showcase presentation and underpin the scoring thresholds in `backend/app/risk_predictor.py`.